In [ ]:
import multiprocessing
import os

os.environ["XLA_FLAGS"] = "--xla_force_host_platform_device_count={}".format(
    multiprocessing.cpu_count()
)

import batman
import blackjax
from astropy.table import Table
import gpjax as gpx
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from scipy.optimize import minimize
import numpy as np
import pandas as pd
from gpjax.kernels import RBF, Linear, Periodic
from jax import jit
from jaxoplanet import orbits
from jaxoplanet.light_curves import LimbDarkLightCurve
from jaxopt import ScipyMinimize
from tensorflow_probability.substrates.jax.bijectors import Log
from tensorflow_probability.substrates.jax.distributions import (
    Normal,
    TruncatedNormal,
    TransformedDistribution,
    Uniform,
)

from inference import log_likelihood_function
from kernels import OrnsteinUhlenbeck
from kernelsearch import KernelSearch, describe_kernel, get_trainables, set_trainables
from mcmc import nuts_warmup, run_mcmc
from util import calculate_example_lightcurve, lc_model

jax.config.update("jax_enable_x64", True)

rng_key = jax.random.PRNGKey(42)

## Load Data

In [14]:
df = (
    Table.read("../data/external/HATS_46b.fit")
    .to_pandas()
    .drop(columns=["FWB20", "e_FWB20"])  # not used in paper
)

t = df["Time"].to_numpy()
t_min = np.amin(t)
t -= t_min

# white lightcurve
white_lc = df["FWL"].to_numpy().T
white_lc_err = df["e_FWL"].to_numpy().T

# spectroscopic light curves
y = df.iloc[:, 3::2].to_numpy().T
yerr = df.iloc[:, 4::2].to_numpy().T

# mask out transit
mask = np.ones_like(t, dtype=bool)
mask[7:41] = False

# reference parameter and limb darkening parameter
reference = pd.read_csv("../data/external/HATS_46b_reference.csv")
# u1_uncertainties = [0.02] * 6 + [0.01] * 19

## Get GP models for all light curves

In [ ]:
kernel_library = [
    Linear(),
    RBF(),
    OrnsteinUhlenbeck(),
    Periodic(),
]

In [ ]:
gps = {}
for i in range(len(y)):
    D = gpx.Dataset(x[mask].reshape(-1, 1), y[i][mask].reshape(-1, 1))

    tree = KernelSearch(
        kernel_library,
        X=jnp.array(x[mask]),
        y=jnp.array(y[i][mask]),
        obs_stddev=jnp.amax(yerr[i][mask]),
        verbosity=0,
    )

    model = tree.search(
        depth=7,
        n_leafs=2,
        patience=0,
    )
    print(describe_kernel(model))
    gps[i] = model

## Define LC model

In [ ]:
def get_lc_model(ld_pars):
    def lc_model(t, params):
        central = orbits.keplerian.Central(
            mass=0.869,
            radius=0.8132,  # the radius in solar radii calculated as (0.05272 AU)/13.94, where 13.94 is the best fit a/R_star given in the paper
        )

        # The light curve calculation requires an orbit
        orbit = orbits.keplerian.Body(
            central=central,
            period=4.7423749,
            radius=params[0] * central.radius,
            inclination=jnp.deg2rad(87.6),
            time_transit=57983.20725000 - x_min,
        )

        lc = LimbDarkLightCurve([params[1], ld_pars[1]]).light_curve(orbit, t=t)
        return lc

    return lc_model

## Define likelihood, prior, posterior

In [ ]:
def get_logprob(model, y, lc_model, ld_params, ld_params_unc):
    initial_position = {
        "gp_parameter": get_trainables(model, unconstrain=True),
        "lc_parameter": jnp.array([0.11250, ld_params[0]]),
    }

    param_priors = {
        "gp_parameter": Normal(
            loc=initial_position["gp_parameter"],
            scale=0.1 * jnp.abs(initial_position["gp_parameter"]),
        ),
        "lc_parameter": Normal(
            loc=initial_position["lc_parameter"],
            scale=[0.06, ld_params_unc],
        ),
    }

    log_likelihood = log_likelihood_function(
        model,
        lc_model,
        x,
        y,
        mask,
        fix_gp=False,
        compile=True,
    )

    @jit
    def log_priors(params):
        gp_log_priors = param_priors["gp_parameter"].log_prob(params["gp_parameter"])
        lc_log_priors = param_priors["lc_parameter"].log_prob(params["lc_parameter"])
        return jnp.sum(gp_log_priors) + jnp.sum(lc_log_priors)

    @jit
    def log_probability(params):
        return log_likelihood(params) + log_priors(params)

    return log_probability, initial_position

## Fits

In [ ]:
sols = []
for i in range(len(y)):
    lc_model = jit(get_lc_model(ld_pars[i]))
    log_probability, initial_position = get_logprob(
        gps[i],
        y[i],
        lc_model,
        ld_pars[i],
        u1_uncertainties[i],
    )

    lbfgsb = ScipyMinimize(
        fun=jit(lambda par: -log_probability(par)), method="l-bfgs-b"
    )
    lbfgsb_sol = lbfgsb.run(initial_position)
    sol = lbfgsb_sol.params
    print(sol["lc_parameter"])
    sols.append(sol)

## Do MCMC

In [ ]:
# Adapted from BlackJax's introduction notebook.
num_adapt = 400
num_samples = 400
num_chains = 8

chains = []

for i in range(len(y)):
    lc_model = jit(get_lc_model(ld_pars[i]))
    log_probability, initial_position = get_logprob(
        gps[i], y[i], lc_model, ld_pars[i], u1_uncertainties[i]
    )

    rng_key, warmup_key, sample_key = jax.random.split(rng_key, 3)

    state, parameters = nuts_warmup(
        warmup_key,
        log_probability,
        initial_position,
        num_steps=num_adapt,
    )

    initial_positions = {
        "gp_parameter": jnp.tile(state.position["gp_parameter"], (num_chains, 1)),
        "lc_parameter": jnp.tile(state.position["lc_parameter"], (num_chains, 1)),
    }

    final_state, state_history, info_history = run_mcmc(
        sample_key,
        log_probability,
        parameters,
        initial_positions,
        num_steps=num_samples,
    )

    chain = np.array(
        state_history.position["lc_parameter"].reshape(
            -1, state_history.position["lc_parameter"].shape[-1]
        )
    )
    chains.append(chain)